# Create OTKA / NKFIH (Hungary) Awards

**One harvest, one provenance (`otka_nkfih`, priority `429`), TWO funder entities split era-wise (runbook §2.3.2).**
Source: the NKFI-EPR public basic-research project database at
http://nyilvanos.otka-palyazat.hu/ — covers OTKA/NKFIH-funded projects that ended in 2005
or later plus all ongoing ones (older grants exist only in print archives). Harvested by
integer-id enumeration of `index.php?menuid=930&num={id}&lang=EN` (the search UI returns no
lists to non-JS clients; ids are sparse — real ids in ~32190..157671). See
`scripts/local/otka_nkfih_to_s3.py`.

**Harvest 2026-07-12: 11,350 projects** (site footer states 11,533 total, so ~98.3%; the
residual is server load-shed flake on this legacy site, not a systematic gap — see the
script docstring). Coverage: title 100% (EN 99.0%, HU 100%); PI, institution, start_date,
end_date all 100%; amount 99.5%; English summary 55.5%. Amounts total ≈267.1 billion HUF.

**Funder attribution (§2.3.2 era split — one source system, two funding-body eras):**

| funder | funder_id | rows |
|--------|-----------|------|
| Hungarian Scientific Research Fund (OTKA) | `4320321994` | **6,316** — projects starting **before 2015-09-01** (classic OTKA grants: K/T/F/PD/NK/NN/... decided by OTKA; incl. the final 2014 OTKA call whose projects started 2015-01..2015-08) |
| Nemzeti Kutatási Fejlesztési és Innovációs Hivatal (NKFIH) | `4320326762` | **5,034** — projects starting **2015-09-01 or later** (NKFIH's first own OTKA-type call was 2015 with starts from 2015-09-01; also all NKFIH-only schemes: FK/KH/KKP post-2015, RGH, EXCELLENCE, STARTING/ADVANCED). Boundary verified: NKFIH-only schemes never appear before 2015-09-01. |

NKFIH legally succeeded OTKA on 2015-01-01 and took over administration of all running
contracts; the split above approximates **which body decided/awarded the grant**, which is
what publications acknowledge. Rows with a NULL start date (rare) fall back to the id
boundary `num >= 115000` (first NKFIH-call ids are 115xxx).

**F4320336675 ("National Research, Development and Innovation Office") is an OpenAlex
DUPLICATE of F4320326762** — same organization (both Wikidata Q30290711; Crossref DOIs
10.13039/501100011019 vs 10.13039/501100018818). Per §2.3.2 we do NOT split rows between
duplicates: all NKFIH-era rows go to F4320326762 (more works: 19.7k vs 14.9k). No rows are
written for F4320336675.

**Amounts — unit handling:** the source publishes "Funding (in million HUF)" (`aktuális
összeg (MFt)`, '.' = decimal separator, e.g. `8.626` = 8,626,000 HUF). The download script
preserves the raw string (`funding_mhuf`) and ships `amount_huf` = value × 1,000,000 in
**whole forints**; this notebook maps `amount_huf` → `amount` with currency `HUF`
(single-country funder, currency implicit — hardcoded and documented here).

**Names:** source lists people as "Family, Given" (Hungarian order, comma-separated on the
EN pages) — split on the comma in the script; §2.4.1 `split_name` (Western "First Last")
does not apply.

Prerequisite: `py -3 scripts/local/otka_nkfih_to_s3.py` (resumable; checkpoint JSONL).
Parquet: `s3://openalex-ingest/awards/otka_nkfih/otka_nkfih_projects.parquet`


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.otka_nkfih_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/otka_nkfih/otka_nkfih_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.otka_nkfih_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.otka_nkfih_raw LIMIT 5;

## Step 2: Create OTKA/NKFIH Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.otka_nkfih_awards
USING delta
AS
WITH
-- Path A: both funders are F4320* and present in openalex.common.funder.
funder_lookup AS (
    SELECT CAST(funder_id AS BIGINT) as funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id IN (4320321994, 4320326762)
),
-- §2.3.2 era split: never blanket-assign one funder to a shared-reporting source.
resolved AS (
    SELECT
        g.*,
        CASE
            WHEN TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') >= DATE'2015-09-01' THEN 4320326762  -- NKFIH era
            WHEN TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') IS NOT NULL       THEN 4320321994  -- OTKA era
            WHEN TRY_CAST(g.num AS INT) >= 115000                          THEN 4320326762  -- NULL-date fallback (first NKFIH-call ids = 115xxx)
            ELSE 4320321994
        END as resolved_funder_id
    FROM openalex.awards.otka_nkfih_raw g
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.identifier)))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title_en), ''), NULLIF(TRIM(g.title_hu), ''),
                 CONCAT('OTKA/NKFIH project ', g.identifier)) as display_name,
        COALESCE(NULLIF(TRIM(g.summary_en), ''), NULLIF(TRIM(g.results_en), ''),
                 NULLIF(TRIM(g.summary_hu), ''), NULLIF(TRIM(g.results_hu), '')) as description,
        f.funder_id,
        g.identifier as funder_award_id,
        TRY_CAST(g.amount_huf AS DOUBLE) as amount,
        CASE WHEN TRY_CAST(g.amount_huf AS DOUBLE) IS NOT NULL THEN 'HUF' ELSE NULL END as currency,
        struct(CONCAT('https://openalex.org/F', f.funder_id) as id, f.display_name, f.ror_id, f.doi) as funder,
        CASE WHEN UPPER(TRIM(g.type_code)) = 'PD' THEN 'fellowship' ELSE 'grant' END as funding_type,
        NULLIF(TRIM(g.type_code), '') as funder_scheme,
        'otka_nkfih' as provenance,
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
        CASE WHEN g.pi_family_name IS NOT NULL OR g.pi_given_name IS NOT NULL THEN
            struct(
                NULLIF(TRIM(g.pi_given_name), '') as given_name,
                NULLIF(TRIM(g.pi_family_name), '') as family_name,
                CAST(NULL AS STRING) as orcid,
                TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as role_start,
                CASE WHEN g.institution IS NOT NULL THEN
                    struct(TRIM(g.institution) as name, 'Hungary' as country,
                           CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids)
                ELSE CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>) END as affiliation)
        ELSE NULL END as lead_investigator,
        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
        -- participants_raw is a JSON array of "Family, Given" strings (source order)
        CASE WHEN g.participants_raw IS NOT NULL AND g.participants_raw NOT IN ('[]', '') THEN
            TRANSFORM(
                FILTER(from_json(g.participants_raw, 'array<string>'), p -> NULLIF(TRIM(p), '') IS NOT NULL),
                p -> struct(
                    CASE WHEN instr(p, ',') > 0 THEN NULLIF(TRIM(SUBSTRING_INDEX(p, ',', -1)), '') ELSE CAST(NULL AS STRING) END as given_name,
                    CASE WHEN instr(p, ',') > 0 THEN NULLIF(TRIM(SUBSTRING_INDEX(p, ',', 1)), '') ELSE TRIM(p) END as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>) as affiliation))
        ELSE CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) END as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.identifier)))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM resolved g
    JOIN funder_lookup f ON f.funder_id = g.resolved_funder_id
)
SELECT *
FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw WHERE provenance = 'otka_nkfih' AND priority = 429;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id, amount, currency, funder,
       funding_type, funder_scheme, provenance, start_date, end_date, start_year, end_year,
       lead_investigator, co_lead_investigator, investigators, landing_page_url, doi,
       works_api_url, created_date, updated_date,
       429 as priority  -- OTKA/NKFIH priority (see CreateAwards.ipynb registry)
FROM openalex.awards.otka_nkfih_awards;

## Verification

In [ ]:
%sql
-- §6.1 / §6.3 / §6.7: counts, coverage, amount sanity (expect >50% amounts — published for ~100%)
SELECT COUNT(*) total,
       COUNT(DISTINCT funder_award_id) uniq_award,
       COUNT(DISTINCT id) uniq_id,
       SUM(CASE WHEN display_name IS NULL OR LENGTH(TRIM(display_name))=0 THEN 1 ELSE 0 END) blank_title,
       ROUND(COUNT(description) * 100.0 / COUNT(*), 1) pct_description,
       ROUND(COUNT(lead_investigator.family_name) * 100.0 / COUNT(*), 1) pct_pi,
       ROUND(COUNT(lead_investigator.affiliation.name) * 100.0 / COUNT(*), 1) pct_institution,
       ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) pct_start_date,
       ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) pct_amount,
       COUNT(DISTINCT currency) distinct_currencies,
       MIN(amount) min_amount, MAX(amount) max_amount, ROUND(AVG(amount), 0) avg_amount,
       ROUND(SUM(amount) / 1e9, 1) total_bn_huf
FROM openalex.awards.otka_nkfih_awards;

In [ ]:
%sql
-- §6.5 funder consistency: era split — both funders must have plausible counts,
-- neither may swallow the other (blanket-CROSS-JOIN symptom)
SELECT funder.display_name, funder_id, MIN(start_year) min_yr, MAX(start_year) max_yr, COUNT(*) n
FROM openalex.awards.otka_nkfih_awards
GROUP BY funder.display_name, funder_id
ORDER BY n DESC;

In [ ]:
%sql
-- §6.6 year distribution (expect ~2001..2026, nothing outside)
SELECT start_year, COUNT(*) cnt
FROM openalex.awards.otka_nkfih_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;

In [ ]:
%sql
-- §6.4a PI / title frequency long-tail check (catches systematic scraper bugs).
-- PI re-grants exist (same PI, several projects over 20 yrs) so a family/given combo
-- may reach ~5-10 rows; hundreds = scraper bug.
SELECT lead_investigator.given_name given, lead_investigator.family_name family, COUNT(*) n
FROM openalex.awards.otka_nkfih_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) n
FROM openalex.awards.otka_nkfih_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- scheme codes × era: classic OTKA schemes (T/F/K/NK/NN/PD...) should sit pre-2015,
-- NKFIH-only schemes (FK/KH/KKP/RGH/...) post-2015
SELECT funder_scheme, funder_id, MIN(start_year) min_yr, MAX(start_year) max_yr, COUNT(*) n
FROM openalex.awards.otka_nkfih_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 30;

In [ ]:
%sql
-- §6.8 confirm rows reached the shared raw table
SELECT provenance, priority, COUNT(*) n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'otka_nkfih'
GROUP BY provenance, priority;